# Mamba Design

Designing Mamba architecture based on this [block diagram](https://www.researchgate.net/figure/The-Mamba-block-architecture-is-constructed-based-on-SSMs-mathematical-formulation-in_fig1_383792530). 

(1) SSM Equation

$h'(t) = \mathbf{A}h(t) + \mathbf{B}x(t) \\
y(t) = \mathbf{C}h(t)$


In [105]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim

Before, we had to create our own custom tokenizer, based on each char in our vocab_size. We will no longer do this! Let's use a pre-trained tokenizer from **HuggingFace** with all the fun bells and whistles.

In [106]:
from transformers import AutoTokenizer # HuggingFace 

# mamba uses "EleutherAI/gpt-neox-20b" tokenizer
tokenizer_name = "EleutherAI/gpt-neox-20b"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
tokenizer.model_max_length = 2048 # max seq length of Mamba (example)
tokenizer.pad_token = tokenizer.eos_token # explicitly say when things end
test_input = "Hello world!"
tensor_input = tokenizer(test_input, return_tensors="pt")
print(tensor_input["input_ids"].shape) # 1 x L
print(tokenizer.decode(tensor_input["input_ids"][0]))
print(f"Vocab size: {tokenizer.vocab_size}")

torch.Size([1, 3])
Hello world!
Vocab size: 50254


We will also test out the train-test split and get_batch standards in **HuggingFace**. Should be of good use to us long down the road when we want to quickly set these features up for our model of choice.

In [107]:
from datasets import Dataset, load_dataset

# hardcoded set of strings (data)
# raw_data = {
#     "text": [
#         "PyTorch makes tensor math easy.",
#         "Hugging Face standardizes the data pipelines.",
#         "NanoGPT is a great learning resource.",
#         "Attention is all you need."
#     ]
# }
# hf_dataset = Dataset.from_dict(raw_data)
# print(hf_dataset)

# now, we will use load_dataset to get whatever large chunk of data we want for pretraining!
# if we don't shuffle, we're going to go through the same row group A -> row group B -> ...
raw_data_stream = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", streaming=True)
# shuffle from the large dataset for a little bit of randomness :)
# buffer_size says "hey, i want to pre-install 10,000 rows of data and put them into DRAM"
shuffled_stream = raw_data_stream.shuffle(buffer_size=10000)
print(shuffled_stream)

IterableDatasetDict({
    train: IterableDataset({
        features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
        num_shards: 14
    })
})


In [108]:
# take(100) says "hey, out of the entire buffer, i want to stop at just 100 rows if i iterate on it"
# for pretraining, we will rely on the ENTIRE pool of 10k. for now, let's do a small test w/ 100
raw_data = list(shuffled_stream['train'].take(100))
print(raw_data)
print(len(raw_data))

[{'text': '2008 Italian political crisis\nOn 24 January 2008 Prime Minister of Italy Romano Prodi lost a vote of confidence in the Senate by a vote of 161 to 156 votes, causing the downfall of his government. Prodi\'s resignation led President Giorgio Napolitano to request the president of the Senate, Franco Marini, to assess the possibility to form a caretaker government. The other possibility would have been to call for early elections immediately. Marini acknowledged impossibility to form an interim government due to the unavailability of the centre-right parties, and early elections were scheduled for 13 April and 14 April 2008.\nProdi had at the time been in office for 20 months, after winning the elections of April 2006. In February 2007, the Prime Minister handed in his resignation, only to be asked to remain by the President, and winning a vote of confidence in the Parliament.\nThe coalition on which Prodi had built his government, called The Union, consisted of a large number 

In [109]:
# convert this list of dictionaries into a dataset
hf_dataset = Dataset.from_list(raw_data)
print(hf_dataset)

Dataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
    num_rows: 100
})


In [110]:
# train-test split
hf_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(hf_split)
print(hf_split['train']['text'][0]) # one row from our text

DatasetDict({
    train: Dataset({
        features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
        num_rows: 90
    })
    test: Dataset({
        features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score'],
        num_rows: 10
    })
})
Brain training games are no better at training your brain than browsing the internet, according to research going into a new BBC programme.
"Can You Train Your Brain? - A Bang Goes The Theory Special", which airs tonight at 9pm BST on BBC One in the UK, followed 11,430 people over six weeks to measure the impact of brain training activities.
Participants were split into three groups, a third of whom did regular sessions to test reasoning, planning and problem-solving, while a third played specially-created games designed to train short-term memory, attentions, maths and visuospatial skills, and the other third did web-br

In [111]:
# getting a batch with collator (this time for causal LLM)
from transformers import DataCollatorForLanguageModeling # define get_batch
from torch.utils.data import DataLoader # use get_batch 

def tokenize(dataset : Dataset):
    encode = lambda dataset : tokenizer(dataset['text'], truncation=True, pad_to_multiple_of=8, padding=True)
    return dataset.map(encode, batched=True) # batched -> parallel processing

# HF uses the map function to add tokenized input to dataset (with map)
res = tokenize(hf_split['train']) 
print(res)
print(res['text'][0])
print(res['input_ids'][0])
print(len(res['input_ids'][0])) # great, now we did some pre-processing of our input data!

Map: 100%|██████████| 90/90 [00:00<00:00, 1375.27 examples/s]

Dataset({
    features: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score', 'input_ids', 'attention_mask'],
    num_rows: 90
})
Brain training games are no better at training your brain than browsing the internet, according to research going into a new BBC programme.
"Can You Train Your Brain? - A Bang Goes The Theory Special", which airs tonight at 9pm BST on BBC One in the UK, followed 11,430 people over six weeks to measure the impact of brain training activities.
Participants were split into three groups, a third of whom did regular sessions to test reasoning, planning and problem-solving, while a third played specially-created games designed to train short-term memory, attentions, maths and visuospatial skills, and the other third did web-browsing stuff.
Apparently nobody exhibited any more brain power afterwards, although the game-players did get better at the games they were playing.
"Statistically, there are no significa

In [112]:
# now, we need to "clean up" the data so it's compatible with mamba
# make sure when we convert to tensor it adheres to the tokenizer rules we established earlier
# this includes max length, proper padding in tensors, sizing, end-of-sequence char, etc.
hf_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
loader = DataLoader(res['input_ids'], collate_fn=hf_collator, batch_size=8, shuffle=True)
# this should result in a batch from our training/testing!
batch_example = next(iter(loader))['input_ids'] # (B, L)
print(batch_example[0])
print(tokenizer.decode(batch_example[0]))
print(batch_example.shape)

tensor([8497,  326,  247,  ...,    0,    0,    0])
Note that a lunar eclipse can only occur when the moon is in the FULL phase. It represents the blockage of the sun's light by the Earth. Thus the Earth is in between the moon and the sun. Here the moon passes through the shadow cast by the Earth.
In addition though, the moon can shadow the sun's light as viewed from the Earth, and thus the Earth passes through the shadow of the moon, blocking the sun. this is a SOLAR ECLIPSE. Again, the small tilt of the moon's orbit with respect to the plane of the ecliptic and the small eccentricity of the lunar orbit make such eclipses much less common than they would be otherwise, but partial or eclipses are frequent (both solar and lunar).
The next total solar eclipse will be June 21 2001, with the PATH OF TOTALITY crossing South Africa and Madagascar.
|Geometry of solar eclipses|
The arrows point to the location of the Earth for the various types of eclipses.
The shadow cast by the moon can be di


(2) ZOH Discretization

$\bar{\mathbf{A}} = \exp(\Delta \mathbf{A}) \\
\bar{\mathbf{B}} = (\Delta \mathbf{A})^{-1} (\exp(\Delta \mathbf{A}) - \mathbf{I}) \cdot \Delta \mathbf{B}$

(3) Discrete SSM

$h_t = \bar{\mathbf{A}}h_{t-1} + \bar{\mathbf{B}}x_t \\
y_t = \mathbf{C}h_t$


(4) Selective Mechanism

$\mathbf{B}_t = \text{Linear}_B(x_t) \\
\mathbf{C}_t = \text{Linear}_C(x_t) \\
\Delta_t = \text{softplus}(\text{Linear}_\Delta(x_t)) \\
\bar{\mathbf{A}}_t, \bar{\mathbf{B}}_t = \text{discretize}(\Delta_t, \mathbf{A}, \mathbf{B}_t)$